# Medical Assistant: Retrieval-Augmented Generation (RAG) over the Merck Manual

**Author:** Michael "Alex" Herrera  |  UT Austin McCombs Post Graduate Program in AI & Machine Learning (Mar 2026)

**Goal:** Build a question-answering assistant that gives clinicians fast, grounded answers from a 4,000+ page medical reference instead of relying on an LLM's general training data.

**Approach**
1. Baseline: ask an open-source LLM (Mistral-7B-Instruct, quantized GGUF via `llama-cpp-python`) clinical questions directly
2. Prompt engineering: add a system prompt to constrain tone and structure
3. RAG pipeline: load the PDF (PyMuPDF), chunk it (512-token chunks), embed it (`thenlper/gte-large`), store it in a Chroma vector database, and retrieve the top-k passages as context for each question
4. Generation parameter tuning (`temperature`, `top_p`, `top_k`, `k`)
5. Evaluation: LLM-as-a-judge scoring of **groundedness** and **relevance** on a 1-5 scale

**Tools:** Python, LangChain, ChromaDB, Sentence-Transformers, Hugging Face Hub, llama-cpp-python, tiktoken

> Runs on a Google Colab GPU runtime (T4 or better). The Merck Manual PDF is not included in this repository.

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

In [2]:
!pip install -q huggingface_hub==0.35.3 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 langchain-text-splitters chromadb==1.1.1 sentence-transformers==5.1.1

**Note:** After running the install cells above, restart the runtime and run all cells from the next cell down.

In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [2]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [3]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)


In [ ]:
#uncomment the below snippet of code if the runtime is connected to GPU.
#llm = Llama(
#    model_path=model_path,
#    n_ctx=2300,
#    n_gpu_layers=36,
#    n_batch=512
#)

In [ ]:
#uncomment the below snippet of code if the runtime is connected to CPU only.
#llm = Llama(
#    model_path=model_path,
#    n_ctx=1024,
#    n_cores=-2
#)

In [4]:
# CPU-only fallback (use this if the GPU build of llama-cpp-python is unavailable)
from llama_cpp import Llama
import os

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=os.cpu_count()
)

#### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
response("What treatment options are available for managing hypertension?")

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5)

## Question Answering using LLM with Prompt Engineering

A system prompt is prepended to each question to steer the model toward precise, guideline-aligned answers.

In [ ]:
system_prompt = """
You are an AI assistant specializing in medical knowledge. Provide clear, precise, and medically reliable
responses based on established clinical guidelines and best practices.
Structure answers with bullet points where helpful (symptoms, causes, treatments).
If a question requires reference material beyond general medical knowledge, acknowledge the limitation rather than speculating.
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = system_prompt+"\n"+"What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = system_prompt+"\n"+"What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = system_prompt+"\n"+"What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = system_prompt+"\n"+"What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5)

## Data Preparation for RAG

### Loading the Data

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

In [7]:
# Connect Google Drive so the notebook can read the Merck Manual PDF
from google.colab import drive
drive.mount('/content/drive')

pdf_path = "/content/drive/MyDrive/AIML_Projects/rawdata/medical_diagnosis_manual.pdf"

Mounted at /content/drive


In [8]:
pdf_loader = PyMuPDFLoader(pdf_path)

In [9]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}")
    print(manual[i].page_content[:1000], "\n")

#### Checking the number of pages

In [ ]:
len(manual)

4114

### Data Chunking

In [5]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)

In [10]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [11]:
len(document_chunks)

8497

In [ ]:
document_chunks[0].page_content

### Embedding

In [12]:
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

In [13]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [14]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [ ]:
# Preview the first 5 values of each embedding vector
embedding_1[:5], embedding_2[:5]

([-0.011196034960448742,
  -0.009513692930340767,
  -0.012064714916050434,
  -0.01841120421886444,
  0.010811341926455498],
 [-0.011196034960448742,
  -0.009513692930340767,
  -0.012064714916050434,
  -0.01841120421886444,
  0.010811341926455498])

### Vector Database

In [15]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [16]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [17]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

In [18]:
vectorstore.embeddings

In [19]:
vectorstore.similarity_search("Type 2 diabetes management",k=3)

### Retriever

In [20]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [21]:
rel_docs = retriever.get_relevant_documents("What are treatment options for rheumatoid arthritis?")
rel_docs

In [22]:
model_output = llm(
      "What are the treatment options for rheumatoid arthritis?",
      max_tokens=500,
      temperature=0.3,
    )

In [23]:
model_output['choices'][0]['text']

'\n\nRheumatoid arthritis (RA) is a chronic inflammatory disease that primarily affects the joints, although it can also impact other body systems. The goal of RA treatment is to reduce symptoms, prevent joint damage, and improve overall function and quality of life.\n\nThe following are common treatment options for rheumatoid arthritis:\n\n1. Medications:\n- Disease-modifying antirheumatic drugs (DMARDs): These medications can slow down the progression of RA by reducing inflammation and preventing joint damage. Examples include methotrexate, sulfasalazine, leflunomide, hydroxychloroquine, and biologic DMARDs such as adalimumab, etanercept, and infliximab.\n- Corticosteroids: These medications can help reduce inflammation and pain in the short term. They are often used in combination with DMARDs to achieve better control of symptoms. Examples include prednisone and cortisone injections.\n- Nonsteroidal anti-inflammatory drugs (NSAIDs): These medications can help relieve pain and reduce

### System and User Prompt Template

In [24]:
qna_system_message = """
You are a highly accurate, professional medical assistant specialized in the content of the Merck Manual of Diagnosis and Therapy (19th Edition).

Your only source of knowledge is the provided context excerpts from the Merck Manual.
- Answer only using information explicitly present in the given context.
- Do not use any external knowledge, general internet information, or your pre-training data.
- If the question cannot be fully answered from the provided context, say: "I'm sorry, but the available Merck Manual excerpts do not contain enough information to answer this question accurately."
- Be concise, factual, objective, and professional.
- Avoid speculation, personal opinions, or advice that could be interpreted as medical diagnosis/treatment without physician consultation.
- Structure answers clearly: use bullet points or numbered lists when listing symptoms, treatments, causes, etc.
- When possible, include key section references or headings from the context.
"""

In [25]:
qna_user_message_template = """
Context (excerpts from Merck Manual 19th Edition):
{context}

User question:
{question}

Answer the question based only on the context above. Be precise and include relevant details.
"""

### Response Function

In [26]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = vectorstore.similarity_search(user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,top_k=20)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,top_k=20)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,top_k=20)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,top_k=20)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,top_k=20)

### Generation Parameter Tuning

The model weights are not changed here. Instead, each query is re-run with different generation settings (`temperature`, `top_p`, `top_k`, `k` retrieved chunks, `max_tokens`) to see how they affect answer completeness and faithfulness to the manual.

In [ ]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,temperature=0.5)

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,temperature=0.5)

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,temperature=0.5)

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,temperature=0.5)

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,temperature=0.5)

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [27]:
groundedness_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users.
You will be presented with a question, the context used by the AI system to generate the answer, and the AI-generated answer.
In the input, the question begins with ###Question, the context begins with ###Context, and the answer begins with ###Answer.

Evaluation criterion (Groundedness): the extent to which the answer is derived ONLY from the provided context,
without adding facts that are not present in it.

Rate on a 1-5 scale:
1 - The answer is not grounded in the context at all
2 - The answer is grounded in the context only to a limited extent
3 - The answer is grounded in the context to a good extent
4 - The answer is grounded in the context mostly
5 - The answer is fully grounded in the context

Respond in this format:
Score: <1-5>
Explanation: <one or two sentences justifying the score>
"""

In [28]:
relevance_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users.
You will be presented with a question, the context used by the AI system to generate the answer, and the AI-generated answer.
In the input, the question begins with ###Question, the context begins with ###Context, and the answer begins with ###Answer.

Evaluation criterion (Relevance): the extent to which the answer directly and completely addresses
every part of the question, without drifting into unrelated information.

Rate on a 1-5 scale:
1 - The answer is not relevant to the question at all
2 - The answer is relevant only to a limited extent
3 - The answer is relevant to a good extent
4 - The answer is mostly relevant
5 - The answer is fully relevant and addresses every part of the question

Respond in this format:
Score: <1-5>
Explanation: <one or two sentences justifying the score>
"""

In [29]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [30]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = vectorstore.similarity_search(user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",max_tokens=200)

print(ground,end="\n\n")
print(rel)

 Score: 5
Explanation: The answer is fully grounded in the context as it accurately summarizes and quotes key information from the provided text regarding the management of sepsis in a critical care unit, including the importance of prompt empiric antibiotic therapy, antibiotic selection guidelines, and specific antibiotic regimens.

 Score: 5
Explanation: The AI-generated answer directly and completely addresses every part of the question by summarizing the key points from the context regarding the management of sepsis in a critical care unit, including prompt empiric antibiotic therapy, antibiotic selection, and specific antibiotic regimens.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [31]:
ground,rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",max_tokens=200)

print(ground,end="\n\n")
print(rel)

 Score: 5
Explanation: The AI-generated answer accurately summarizes the common symptoms of appendicitis from the context and correctly states that surgical removal is required for treatment. It also mentions the typical age group affected by appendicitis and the role of imaging studies in diagnosis, all of which are present in the context. Additionally, it clarifies that medicine alone cannot cure appendicitis, and if the appendix has perforated, antibiotics should be continued until certain conditions are met. The answer also mentions the possibility of resection of the entire mass when a large inflammatory mass is found, which is consistent with the context.

 Score: 5
Explanation: The AI-generated answer directly addresses every part of the question by listing the common symptoms of appendicitis and stating that surgical removal is required for treatment. It also mentions that antibiotics are used in conjunction with surgery and provides additional information about the use of imag

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [32]:
ground,rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",max_tokens=200)

print(ground,end="\n\n")
print(rel)

 Score: 5
Explanation: The answer is fully grounded in the context provided, as it accurately describes alopecia areata as a condition characterized by sudden patchy hair loss and mentions effective treatments such as topical corticosteroids, intralesional corticosteroid injections, and topical minoxidil, all of which are mentioned in the context.

 Score: 5
Explanation: The answer directly addresses the question by providing a clear explanation of what alopecia areata is and its causes. It also lists effective treatments for this condition, including topical corticosteroids, intralesional corticosteroid injections, and topical minoxidil.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [33]:
ground,rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",max_tokens=200)

print(ground,end="\n\n")
print(rel)

 Score: 5
Explanation: The AI-generated answer is fully grounded in the context as it accurately summarizes and repeats the recommended treatments for a person with traumatic brain injury directly from the provided context.

 Score: 5
Explanation: The AI-generated answer directly and completely addresses every part of the question by summarizing the recommended treatments for a person with traumatic brain injury (TBI) according to the Merck Manual, including initial treatment, supportive care, and seizure prevention.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [34]:
ground,rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",max_tokens=200)

print(ground,end="\n\n")
print(rel)

 Score: 5
Explanation: The AI-generated answer is fully grounded in the context provided, as it directly quotes and summarizes information from the context regarding necessary precautions and treatment steps for a person with a fractured leg.

 Score: 5
Explanation: The AI-generated answer directly addresses every part of the question by summarizing the necessary precautions and treatment steps for a person with a fractured leg, including elevating the limb, applying warmth after two days, immobilization using a cast or splint, keeping the cast dry, inspecting it regularly, seeking medical care if needed, and considering the use of a splint instead of a cast for stable injuries.


### Evaluation Summary

| Query | Groundedness (1-5) | Relevance (1-5) |
|---|---|---|
| 1. Sepsis protocol | 5 | 5 |
| 2. Appendicitis | 5 | 5 |
| 3. Patchy hair loss | 5 | 5 |
| 4. Traumatic brain injury | 5 | 5 |
| 5. Leg fracture | 5 | 5 |

The judge's explanations point to specific content carried over from the retrieved passages (e.g., empiric antibiotic regimens for sepsis; topical/intralesional corticosteroids and minoxidil for alopecia areata), which suggests the grounding prompt kept answers anchored to the manual.

**Limitation:** the same Mistral model generates and judges the answers, so uniformly perfect scores are likely optimistic. A stronger or independent judge model, or clinician review of a sample, would give a more reliable estimate.

## Actionable Insights and Business Recommendations

Insights:

- The RAG system successfully reduces information overload by retrieving only the most relevant information from the Merck Manual.
- Clinical decision making can now be faster by delivering quick, relevant answers to medical questions.
- Grounding answers in a trusted reference, and running an open-source model privately, makes the system easier to audit than a general-purpose chatbot, though clinician review is still required before any clinical use.
- Having faster access to this knowledge has the potential to improve patient outcomes through earlier and more accurate delivery of information.


Improvements:

- Add updates from newer editions of manual or supplemental information
- Collect usage data and clinician feedback to identify weak areas and improve like chunking or fine tune more.
- Can build specialization variants of the system (cardiology, emergency medicine, pediatrics, oncology, etc.) based on of this.